[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_dae_top3.ipynb)

# Part A: the final DAE, the top 3 for 100 epochs with 3 seeds

Runs after `colab_dae_h.ipynb`, from its choice (`"dae"` in `tuning_choice.json`). Every run on all the images, by **val k-NN top-1** (the best epoch is kept).

**Stage A, last checks (20 epochs)**, one change at a time from the choice: h 128 and 64 (only larger h was tried, and 384 lost to 256), a second noise (Gaussian 0.1 and 0.2 on top of salt and pepper; it helped patch_mask), batch 256 and weight decay 0.05. 6 runs, about 1.2 hours.

**Stage B, the top 3 for 100 epochs with seeds 42, 1 and 2.** The 3 best of every full-data DAE run (`colab_dae_full.ipynb`, `colab_dae_h.ipynb`, stage A), each for 100 epochs with 3 seeds: 9 runs of about 57 minutes. The first 20 epochs repeat the 20-epoch runs; the next 80 show whether val k-NN rises again after its peak. The seeds give a mean and a standard deviation, so the choice is not decided by noise.

**Three runtimes.** `PART` = 1, 2, 3 splits stage B (3 runs each, about 2.9 hours); runtime 1 also runs stage A first, so start runtimes 2 and 3 when stage A is done. `PART` = 0 runs everything here (about 9.7 hours).

The last cell saves the DAE with the best mean val k-NN over the 3 seeds as `"dae"` (with `num_epochs` 100); the choice of `colab_dae_h.ipynb` is kept as `"dae_h_final"`.

## Setup
Downloads the repository (or pulls the latest code), installs it and checks the GPU.

In [12]:
import os

REPO = "/content/Representation-Learning-Midterm-DAE-SimCLR"
if not os.path.exists(REPO):
    !git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git {REPO}
os.chdir(REPO)
!git pull
!git log --oneline -1

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 10 (delta 7), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 18.04 KiB | 2.25 MiB/s, done.
From https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR
   65b6c22..0713ef3  main       -> origin/main
Updating 65b6c22..0713ef3
Fast-forward
 callbacks.py              |   23 +-
 colab_dae_h.ipynb         |  257 -------
 colab_dae_top3.ipynb      |  318 +++++++++
 history/colab_dae_h.ipynb | 1723 +++++++++++++++++++++++++++++++++++++++++++++
 4 files changed, 2058 insertions(+), 263 deletions(-)
 delete mode 100644 colab_dae_h.ipynb
 create mode 100644 colab_dae_top3.ipynb
 create mode 100644 history/colab_dae_h.ipynb
0713ef3 (HEAD -> main, origin/main, origin/HEAD) KNNCallback: macro precision, recall, F1 and AUROC of the k-NN on the kept checkpoint


In [13]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), "|", os.cpu_count(), "CPU cores")

NVIDIA L4 | 12 CPU cores


## Drive
Every run writes to `midterm_runs`, next to the earlier runs. The runs here have `tune_pretrain_DAE` in their folder name.

In [14]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/לימודים/תואר שני/שנה ב/סמסטר א/למידת ייצוגים במודלים עמוקים/מטלות/מטלת אמצע"
OUT = f"{DRIVE_DIR}/midterm_runs"
assert os.path.isdir(OUT), f"no folder {OUT}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## TensorBoard
Filter the runs with `tune_pretrain`, then *Toggle all runs*. `epoch/val_knn_top1` shows where each run peaks, `epoch/val_loss` the reconstruction, the TEXT tab each run's `run_info`.

In [15]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 1916), started 19:50:30 ago. (Use '!kill 1916' to kill it.)

<IPython.core.display.Javascript object>

## The runs
`train` runs train.py only if the run has no `results.json` yet: after a disconnect, run all again; a finished run is read from Drive and an unfinished one continues from its `last.ckpt`.

In [16]:
import importlib
import itertools
import json
import sys

import pandas as pd

RUN20 = "--mode pretrain --tune --num_epochs 20"  # all the images, the full k-NN bank, val only (no test)
RUN100 = "--mode pretrain --tune --num_epochs 100"
CHOICE_FILE = f"{OUT}/tuning_choice.json"
choices = json.load(open(CHOICE_FILE))
assert "dae_h" in choices, "run colab_dae_h.ipynb to the end first: its choice is the base here"
base = {k: v for k, v in choices.get("dae_h_final", choices["dae"]).items() if k != "num_epochs"}
print("base:", base)


def flags_of(params):
    """The train.py flags of one DAE."""
    p = dict(params)
    return f"--channels {p.pop('conv1')} {p.pop('conv2')} " + " ".join(f"--{k} {v}" for k, v in p.items())


def folder_of(flags):
    """The run folder train.py uses for these flags (run_info.folder_name)."""
    import config, utils, run_info
    for module in (config, utils, run_info):
        importlib.reload(module)
    sys.argv = ["train.py", *flags.split(), "--output_dir", OUT]
    return f"{OUT}/{run_info.folder_name(utils.parse_args().mode)}"


def train(flags):
    """One run of train.py (skipped if it finished before): its val numbers and its folder."""
    if not os.path.exists(f"{folder_of(flags)}/results.json"):
        !python train.py {flags} --output_dir "{OUT}"
    return read(flags)


def read(flags):
    """The val numbers of a finished run, or None if it has not finished."""
    folder = folder_of(flags)
    if not os.path.exists(f"{folder}/results.json"):
        return None
    with open(f"{folder}/results.json") as f:
        return {**json.load(f), "run": os.path.basename(folder)}

base: {'noise_type': 'salt_pepper', 'noise_level': 0.5, 'patch_size': 8, 'conv1': 32, 'conv2': 64, 'latent_dim': 256, 'dropout': 0.4, 'augment': 'flip', 'batch_size': 512, 'optimizer': 'adamw', 'activation': 'gelu', 'lr': 0.001, 'wd': 0.01, 'scheduler': 'none', 'noise_type2': 'none', 'noise_level2': 0.0}


## Which part
`PART` = 1, 2 or 3 in three runtimes (runtime 1 also runs stage A; start 2 and 3 after it), or 0 for everything here.

In [17]:
PART = 1  # 0: everything here; 1, 2, 3: a third of stage B each (1 also runs stage A)

## Stage A: last checks, 20 epochs
One change at a time from `base`; `base` itself was trained in `colab_dae_h.ipynb` and is read from Drive.

In [18]:
checks = {"base": {}, "h 128": {"latent_dim": 128}, "h 64": {"latent_dim": 64},
          "+ gaussian 0.1": {"noise_type2": "gaussian", "noise_level2": 0.1},
          "+ gaussian 0.2": {"noise_type2": "gaussian", "noise_level2": 0.2},
          "batch 256": {"batch_size": 256}, "weight decay 0.05": {"wd": 0.05}}
if PART in (0, 1):
    rows = []
    for name, change in checks.items():
        r = train(f"{RUN20} {flags_of({**base, **change})}")
        rows.append({"check": name, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"], "run": r["run"]})
    display(pd.DataFrame(rows).sort_values("val_knn", ascending=False).round(4))

,check,val_knn,val_loss,run
6,weight decay 0.05,0.1065,0.2584,date24112026_time164702_CNN3_tune_pretrain_DAE...
0,base,0.1064,0.2525,date24112026_time130158_CNN3_tune_pretrain_DAE...
3,+ gaussian 0.1,0.1045,0.2872,date24112026_time161304_CNN3_tune_pretrain_DAE...
5,batch 256,0.1045,0.2449,date24112026_time163541_CNN3_tune_pretrain_DAE...
4,+ gaussian 0.2,0.1034,0.2908,date24112026_time162419_CNN3_tune_pretrain_DAE...
1,h 128,0.1000,0.3037,date24112026_time155052_CNN3_tune_pretrain_DAE...
2,h 64,0.0845,0.3739,date24112026_time160200_CNN3_tune_pretrain_DAE...


## The top 3
Every full-data DAE run of 20 epochs so far, read from Drive (all of stage A must be done). The 3 best by val k-NN go on to stage B.

In [19]:
b0 = {"noise_type": "salt_pepper", "noise_level": 0.5, "patch_size": 8, "conv1": 32, "conv2": 64,
      "latent_dim": 256, "dropout": 0.0, "augment": "flip", "batch_size": 512, "optimizer": "adamw",
      "activation": "gelu", "lr": 0.001, "wd": 0.01, "scheduler": "none", "noise_type2": "none", "noise_level2": 0.0}
h_winner = {k: v for k, v in choices["dae_h"].items() if k != "num_epochs"}  # stage 1 of colab_dae_h.ipynb
candidates = ([{**b0, "dropout": d, "lr": lr} for d, lr in itertools.product([0.0, 0.1, 0.2, 0.3], [0.001, 0.0005])]
              + [{**b0, "latent_dim": h, "dropout": d} for h, d in itertools.product([256, 384, 1024], [0.3, 0.4, 0.5])]
              + [{**h_winner, **c} for c in ({"noise_level": 0.6}, {"noise_level": 0.7}, {"loss": "l1"}, {"scheduler": "cosine"},
                                              {"noise_level": 0.3, "noise_level_end": 0.7})]
              + [{**base, **c} for c in checks.values()])
rows, missing = [], []
for params in candidates:
    r = read(f"{RUN20} {flags_of(params)}")
    if r is None:
        missing.append(params)
    else:
        rows.append({"params": params, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"], "run": r["run"]})
assert not [p for p in missing if any(p == {**base, **c} for c in checks.values())], "stage A is not done yet"
all20 = pd.DataFrame(rows).drop_duplicates(subset="run").sort_values("val_knn", ascending=False)
display(all20[["val_knn", "val_loss", "run"]].head(10).round(4))
top3 = all20["params"].head(3).tolist()

,val_knn,val_loss,run
28,0.1065,0.2584,date24112026_time164702_CNN3_tune_pretrain_DAE...
9,0.1064,0.2525,date24112026_time130158_CNN3_tune_pretrain_DAE...
6,0.1058,0.2289,date24112026_time112843_CNN3_tune_pretrain_DAE...
20,0.1053,0.2478,date24112026_time152446_CNN3_tune_pretrain_DAE...
27,0.1045,0.2449,date24112026_time163541_CNN3_tune_pretrain_DAE...
25,0.1045,0.2872,date24112026_time161304_CNN3_tune_pretrain_DAE...
4,0.1045,0.2256,date24112026_time110557_CNN3_tune_pretrain_DAE...
12,0.1042,0.2885,date24112026_time133559_CNN3_tune_pretrain_DAE...
11,0.1035,0.2787,date24112026_time132425_CNN3_tune_pretrain_DAE...
26,0.1034,0.2908,date24112026_time162419_CNN3_tune_pretrain_DAE...


## Stage B: the top 3, 100 epochs, seeds 42, 1 and 2
9 runs; `PART` 1, 2 or 3 takes every third one.

In [20]:
tasks = [(i, seed) for i in range(3) for seed in (42, 1, 2)]
for i, seed in (tasks[PART - 1::3] if PART else tasks):
    params = {**top3[i], **({} if seed == 42 else {"seed": seed})}
    print(f"top {i + 1}, seed {seed}")
    train(f"{RUN100} {flags_of(params)}")

top 1, seed 42
top 2, seed 42
top 3, seed 42


## The choice
All 9 runs, read from Drive (all three runtimes must be done): the mean and standard deviation over the 3 seeds. The best mean is saved as `"dae"`.

In [23]:
rows = []
for i, seed in tasks:
    r = read(f"{RUN100} {flags_of({**top3[i], **({} if seed == 42 else {'seed': seed})})}")
    if r is None:  # not finished yet: left out of the table
        print(f"not finished yet: top {i + 1}, seed {seed}")
        continue
    rows.append({"top": i + 1, "seed": seed, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"], "run": r["run"]})
runs100 = pd.DataFrame(rows)
display(runs100.round(4))
summary = runs100.groupby("top")["val_knn"].agg(["mean", "std", "min", "max"]).round(4)
display(summary)
if len(rows) == len(tasks):  # the choice only when all 9 runs are done
    best = int(summary["mean"].idxmax())
    choices.setdefault("dae_h_final", choices["dae"])  # the choice of colab_dae_h.ipynb, kept
    choices["dae"] = {**top3[best - 1], "num_epochs": 100}
    with open(CHOICE_FILE, "w") as f:
        json.dump(choices, f, indent=1)
    print(f"top {best}:", choices["dae"])

not finished yet: top 1, seed 1
not finished yet: top 1, seed 2
not finished yet: top 2, seed 1
not finished yet: top 2, seed 2
not finished yet: top 3, seed 1
not finished yet: top 3, seed 2


,top,seed,val_knn,val_loss,run
0,1,42,0.1059,0.2527,date24112026_time165827_CNN3_tune_pretrain_DAE...
1,2,42,0.1060,0.2588,date24112026_time175148_CNN3_tune_pretrain_DAE...
2,3,42,0.1059,0.2469,date24112026_time184506_CNN3_tune_pretrain_DAE...


,mean,std,min,max
top,,,,
1,0.1059,NaN,0.1059,0.1059
2,0.1060,NaN,0.1060,0.1060
3,0.1059,NaN,0.1059,0.1059
